In [1]:
#!pip install transformers tensorflow

In [2]:
#pip install transformers torch


In [3]:
#pip install torch transformers datasets


In [16]:
#pip install --upgrade torch transformers datasets


In [3]:
import torch

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments


In [5]:
df = pd.read_csv("NLP with disaster tweets.csv")

In [6]:
texts = list(df['text'].values)
labels = list(df['target'].values)

In [7]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

encodings = tokenizer(texts, truncation=True, padding=True, max_length=64)


# Create PyTorch Dataset

In [8]:
class DisasterDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_texts, test_texts, train_labels, test_labels = train_test_split(texts, labels, test_size=0.2)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=64)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=64)

train_dataset = DisasterDataset(train_encodings, train_labels)
test_dataset = DisasterDataset(test_encodings, test_labels)


# Load Pretrained Model

In [9]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.


# Define Training Arguments

In [10]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=10,
)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# Initialize Trainer

In [11]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)


# Train & Evaluate

In [12]:
trainer.train()
trainer.evaluate()


C:\Users\Mehak\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.650043
20,0.497387
30,0.435245
40,0.416591
50,0.512422
60,0.460609
70,0.427783
80,0.452417
90,0.491398
100,0.499299


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Mehak\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Mehak\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Mehak\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
0.087460,0.515311,1143


{'eval_loss': 0.5153114199638367}

# Make Predictions

In [20]:
sample_text = ["I love spending my weekends watching movies with friends!"]
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True, max_length=64)
outputs = model(**inputs)
pred = torch.argmax(outputs.logits, dim=1).item()

print("Disaster Tweet" if pred == 1 else "Not Disaster Tweet")


Not Disaster Tweet


In [14]:
results = trainer.evaluate()
print(results)


Training Loss,Validation Loss,Step
0.087460,0.515311,1143


{'eval_loss': 0.5153114199638367}


In [19]:
from sklearn.metrics import accuracy_score

# Get predictions
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

# Accuracy
acc = accuracy_score(test_labels, y_pred)
print(f"Test Accuracy: {acc*100:.2f}%")


Test Accuracy: 83.91%
